# Chapter 11. Reading a City Through Text and Images

*Starting With What You Have: A Quantitative Field Guide for Urban Research in Data-Scarce Settings*

Runs in a browser with no installation. Open in Google Colab and choose Runtime, then Run all.


## Step 0. Installation

No GPU and no large model download. The corpus includes a variant reproducing low-resource language conditions, so the cost of that gap can be measured rather than asserted.

In [ ]:
!pip install -q scikit-learn pandas pillow matplotlib

## Step 1. Corpus diagnostics, before anything else

`tokens_per_type` is the measure that matters. The low-resource variant has **more** unique types and fewer examples of each, from the same number of documents. It is not that there is less data; the signal is scattered.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import text_toolkit as tt

en = pd.read_csv("data/complaints_en.csv")
lr = pd.read_csv("data/complaints_lowres.csv")
print("English :", tt.vocab_profile(en.text))
print("Low-res :", tt.vocab_profile(lr.text))

## Step 2. Classification, word against character n-grams

Where spelling wavers, character n-grams are more robust: `water` and `wter` share the trigram `ter`.

In [ ]:
print("English, word n-grams      :", tt.evaluate(en.text, en.label))
print("low-resource, word n-grams :", tt.evaluate(lr.text, lr.label))
print("low-resource, char n-grams :", tt.evaluate(lr.text, lr.label, char_ngrams=True))

## Step 3. The learning curve, the most important experiment here

Labelling is the most expensive task, so this is a research design decision. Read where English saturates, how much more the low-resource condition needs, and **where the gap is widest.**

In [ ]:
ce = tt.learning_curve(en.text.values, en.label.values)
cl = tt.learning_curve(lr.text.values, lr.label.values)
m = ce.merge(cl, on="n_train", suffixes=("_en", "_lowres"))
m["gap"] = (m.f1_macro_en - m.f1_macro_lowres).round(3)
print(m[["n_train", "f1_macro_en", "f1_macro_lowres", "gap"]].to_string(index=False))

## Step 4. Topic modelling, when there are no labels at all

If you do not even know what to label, look at the structure first.

In [ ]:
tp, W, lda = tt.topic_model(en.text.values, n_topics=5)
tp

## Step 5. How well do unsupervised topics match the true categories?

High agreement means that at the exploratory stage, manual labelling can be skipped.

In [ ]:
ct = tt.topic_label_crosstab(W, en.label.values)
print(ct.to_string())
print("agreement:", round(float(ct.values.max(axis=1).sum() / ct.values.sum()), 3))

## Step 6. Street imagery and the Green View Index

An ordinary camera has no near infrared band, so Excess Green is used instead of NDVI. It measures a different quantity from the green cover a satellite sees from above.

In [ ]:
import image_toolkit as it

X, y, dfi = it.build_feature_table("data/streetview_labels.csv")
gi = it.FEATURE_NAMES.index("GVI")
dfi["GVI"] = X[:, gi]
print(dfi.groupby("label")[["true_green", "GVI"]].mean().round(4).to_string())
print("GVI vs planted green, r =",
      round(float(np.corrcoef(dfi.true_green, dfi.GVI)[0, 1]), 4))

## Step 7. One indicator is not enough

Note which feature contributes most; it is not the one you would expect, and it requires no training data to compute.

In [ ]:
print("GVI alone   :", it.evaluate(X[:, [gi]], y))
print("all features:", it.evaluate(X, y))
print()
print(it.importance(X, y).head(5).to_string(index=False))

## Step 8. Handing over to Chapter 4

Aggregate complaints by zone and the result feeds directly into the spatial statistics of Chapter 4.

In [ ]:
agg = en.groupby("label").size().rename("n_complaints")
print(agg.to_string())
print()
print("Attach a zone code, merge onto a GeoJSON, and Chapter 4 applies directly.")

---

**What to do next.** Compare against Section 11.4. Whatever you conclude from complaint data, state the bias: an area with few complaints may be an area without a voice.